## Aufgabe 4.2: Entwurf rekursiver Filter

Ziel dieser Aufgabe ist es, ein digitales Tiefpasssystem basierend auf einem gegebenen Toleranzschema für die Amplitudenfrequenzantwort zu entwerfen. Es stehen Python-Funktionen, z. B. `butter`, `cheby1` usw., aus `scipy.signal` zur Verfügung, um einen Tiefpassentwurf für kontinuierliche Systeme zu realisieren. Das im Z-Bereich gegebene Toleranzschema muss in den S-Bereich transformiert werden. Die bekannten Algorithmen werden verwendet, um den gewünschten Filter im S-Bereich zu entwerfen. Anschließend wird die Systemfunktion zurück in den Z-Bereich transformiert.

Für die Transformation vom Z- in den S-Bereich und umgekehrt wird die **bilineare Transformation** verwendet, die stabile Systeme im S-Bereich in stabile Systeme im Z-Bereich überführt (Abschnitt 4.2.1). Bei der bilinearen Transformation ist eine Vorverzerrung der $\Omega$-Achse im Z-Bereich notwendig. Der Entwurf kann unter Verwendung der Standardverfahren aus Abschnitt 4.2.2 und 4.2.3 mithilfe einer Hilfsfunktion durchgeführt werden.

Für den Tiefpassentwurf stehen in Python (`scipy.signal`) die folgenden Funktionen zur Verfügung. Die Berechnung der notwendigen Filterordnung und der normierten Grenzfrequenz\footnote{Alle Entwurfsfunktionen verwenden normierte Frequenzen zwischen 0 und 1. Die Normierung erfolgt bezogen auf die halbe Abtastfrequenz. Die normierte Grenzfrequenz gibt die Frequenz mit $3 \, \text{dB}$ Dämpfung an. Beispiel: Abtastfrequenz 1000 Hz, Grenzfrequenz 300 Hz, normierte Grenzfrequenz: `Wn=300/500=0.6`} kann mit folgenden Funktionen durchgeführt werden:

| Filtertyp          | Bestimmung der Filterordnung                                  |
| ------------------ | ------------------------------------------------------------ |
| Butterworth        | `buttord(Wp, Ws, Rp, Rs, fs=fs)`                              |
| Tschebyscheff Typ I| `cheb1ord(Wp, Ws, Rp, Rs, fs=fs)`                             |
| Tschebyscheff Typ II| `cheb2ord(Wp, Ws, Rp, Rs, fs=fs)`                            |
| Cauer              | `ellipord(Wp, Ws, Rp, Rs, fs=fs)`                             |

Anschließend können die Koeffizienten mit den folgenden Entwurfsfunktionen berechnet werden:

| Filtertyp          | Entwurfsfunktion                                               |
| ------------------ | ------------------------------------------------------------- |
| Butterworth        | `butter(m, Wn, btype='low', output='ba', fs=fs)`              |
| Tschebyscheff Typ I| `cheby1(m, Rp, Wn, btype='low', output='ba', fs=fs)`          |
| Tschebyscheff Typ II| `cheby2(m, Rs, Wn, btype='low', output='ba', fs=fs)`         |
| Cauer              | `ellip(m, Rp, Rs, Wn, btype='low', output='ba', fs=fs)`       |

Die Bedeutung der Parameter ist wie folgt:

| Parameter          | Bedeutung                                                      |
| ------------------ | ------------------------------------------------------------- |
| `b, a`             | Vektoren mit den Filterkoeffizienten                          |
| `m`                | Ordnung des Filters                                            |
| `Wn`               | Normierte Grenzfrequenz des Tiefpasses                        |
| `Wp`               | Normierte Durchlassfrequenz des Tiefpasses                    |
| `Ws`               | Normierte Sperrfrequenz des Tiefpasses                        |
| `Rp`               | Welligkeit im Durchlassbereich (dB)                           |
| `Rs`               | Dämpfung im Sperrbereich (dB)                                 |

Bestimmen Sie die minimale Filterordnung, die für jeden Filtertyp benötigt wird, und entwerfen Sie die Tiefpassfilter für das folgende Toleranzschema:

- $f_\text{D} = 1 \, \text{kHz}$
- $f_\text{S} = 1{,}4 \, \text{kHz}$
- $R_\text{P} = 0{,}5 \, \text{dB}$
- $R_\text{S} = 30 \, \text{dB}$
- $f_A = 8 \, \text{kHz}$

![Toleranzschema](toleranz.eps)

Vergleichen Sie die Amplitudenfrequenzgänge und die Nullstellen-Pol-Diagramme der verschiedenen Filtertypen miteinander.

In [ ]:
# BEGIN SOLUTION

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import buttord, butter, cheb1ord, cheby1, cheb2ord, cheby2, ellipord, ellip, freqz
from scipy.signal import tf2zpk as zplane  # Berechnung von Nullstellen, Polen und Verstärkung

# Gegebene Frequenzen und Toleranzen
fD = 1000     # Durchlassfrequenz (Hz)
fS = 1400     # Sperrfrequenz (Hz)
Rp = 0.5      # Welligkeit im Durchlassbereich (dB)
Rs = 30       # Dämpfung im Sperrbereich (dB)
fA = 8000     # Abtastfrequenz (Hz)

# Umrechnung in normierte Frequenzen (relativ zur Nyquistfrequenz)
Wp = 2 * fD / fA
Ws = 2 * fS / fA

# Schleife über alle Filtertypen
for typ in range(1, 5):
    if typ == 1:
        n, Wn = buttord(Wp, Ws, Rp, Rs)  # Butterworth
        b, a = butter(n, Wn)
        FTyp = 'Butterworth'
    elif typ == 2:
        n, Wn = cheb1ord(Wp, Ws, Rp, Rs)  # Tschebyscheff Typ I
        b, a = cheby1(n, Rp, Wn)
        FTyp = 'Chebyshev Typ I'
    elif typ == 3:
        n, Wn = cheb2ord(Wp, Ws, Rp, Rs)  # Tschebyscheff Typ II
        b, a = cheby2(n, Rs, Wn)
        FTyp = 'Chebyshev Typ II'
    elif typ == 4:
        n, Wn = ellipord(Wp, Ws, Rp, Rs)  # Cauer (Elliptisch)
        b, a = ellip(n, Rp, Rs, Wn)
        FTyp = 'Cauer'
    else:
        raise ValueError('Etwas ist schiefgelaufen...')

    # Beschriftungstitel
    TitleString = 'Filtertyp: {}  Ordnung: {}'.format(FTyp, n)

    # Amplitudenfrequenzgang berechnen
    W = np.linspace(0, np.pi, 512)
    _, Hg = freqz(b, a, 512)

    # Plot: Amplitudengang
    plt.figure()
    plt.plot(W / np.pi, np.abs(Hg))
    plt.grid(True)
    plt.title(TitleString)
    plt.xlabel(r'$\Omega/\pi$')
    plt.ylabel(r'$|H(e^{j\Omega})|$')
    plt.axis([0, 1, 0, 1.2])

    # Toleranzröhre einzeichnen
    xline = [Wp, Wp]
    yline = [0, 10 ** (-Rp / 20)]
    plt.plot(xline, yline, 'r--')

    xline = [0, Ws]
    yline